# 08 — Girvan–Newman dendrogram on the disease-trajectory graph (issue #14)

Notebook 06 built a graph of **diagnoses**, not patients. An edge A → B means "having A measurably raises the chance of B within a year", among all 6,731 patients. This notebook asks how those diagnoses group into families, using the **Girvan–Newman** algorithm from Lecture 3:

1. Find the edge that the most shortest paths run through (the most "bridge-like" edge) and remove it.
2. Repeat. The graph falls apart into smaller and smaller groups.
3. Record every split. The record of splits *is* a tree, a **dendrogram**: the first split is the top of the tree, the last splits are the leaves.

The guide's original sketch called `next()` once, which gives only the first split into two groups. Here the algorithm runs to the end, and the full sequence is converted into the merge table that `scipy` draws (`ehrgraph.netstats.dendrogram`). Edge weights are relative risks, where bigger means a stronger link, so shortest paths use `1 / relative risk` as the length. That is the same similarity-vs-distance correction as the centrality code.

**Data:** the course's OMOP tables, which every team member has. The tables keep both each diagnosis's original billing code (ICD-9 or ICD-10) and its standard SNOMED concept, so the graph can be built both ways. Readable names come from the vocabulary's concept table when one is available; otherwise concepts are shown by ID.

**What this notebook found, up front:** run on the graph exactly as notebook 06 built it, the tree's top split is **not clinical. It is the 2015 switch from ICD-9 to ICD-10 billing codes.** Mapping both code systems to their shared SNOMED concepts removes the split, and the clinical families appear.

In [ ]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
while not (_repo_root / "pyproject.toml").exists() and _repo_root != _repo_root.parent:
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root))

from scripts.local_setup import ensure_data_sources  # noqa: E402

sources = ensure_data_sources()

In [ ]:
import time
from collections import Counter

import community as community_louvain
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from sklearn.metrics import adjusted_rand_score

from ehrgraph.data.cohort import patient_code_counts
from ehrgraph.data.omop import OmopDatabase, OmopTables, concept_names
from ehrgraph.eval.aggregate import write_aggregate
from ehrgraph.graphs.trajectory import build_population_trajectory_graph, symmetrize_max
from ehrgraph.netstats.dendrogram import girvan_newman_tree
from ehrgraph.viz import dendrogram_figures as df
from ehrgraph.viz import network_figures as nf

GRAPH_VARIANT = "omop_top200_rr365d_support10_bonferroni_symmax"
results_dir = _repo_root / "results" / "aggregate"
figures_dir = results_dir / "figures"
nf.apply_style()

tables = OmopTables(sources.trimmed)

## Part 1 — The graph on raw billing codes (as notebook 06 built it)

Same construction as notebook 06: the 200 most common diagnosis codes, a relative risk within 365 days, at least 10 supporting patients, and Bonferroni correction over all 39,800 ordered pairs. It is then made undirected by keeping the stronger direction, because Girvan–Newman needs an undirected graph. That discards which diagnosis comes first, a tradeoff the report should state.

In [ ]:
t0 = time.time()
raw_db = OmopDatabase(tables, codes="source")  # ICD9CM/... and ICD10CM/... billing codes
G_raw, _ = build_population_trajectory_graph(raw_db)
U_raw = symmetrize_max(G_raw)
pieces_raw = sorted(nx.connected_components(U_raw), key=len, reverse=True)
print(
    f"built in {time.time() - t0:.0f}s:",
    U_raw.number_of_nodes(),
    "diagnoses,",
    U_raw.number_of_edges(),
    "links",
)
print("piece sizes:", [len(p) for p in pieces_raw])

vocab = lambda code: code.split("/")[0]  # noqa: E731
pd.DataFrame(
    [{"piece size": len(p)} | Counter(vocab(c) for c in p) for p in pieces_raw[:2]]
).fillna(0).astype(int)

In [ ]:
cross = sum(1 for u, v in U_raw.edges() if vocab(u) != vocab(v))
print(f"links joining an ICD-9 code to an ICD-10 code: {cross} of {U_raw.number_of_edges()}")

**The two big pieces are the two coding systems.** One is entirely ICD-9, used before October 2015. The other is entirely ICD-10, used after. Not a single link joins them. A one-year window almost never spans the switch, so an ICD-9 diagnosis can only ever "lead to" other ICD-9 codes. The pieces repeat the same clinical story twice: ICD-9 "unspecified septicemia" and ICD-10 "sepsis, unspecified organism" sit in different pieces.

The Girvan–Newman tree for this graph shows it directly. Leaves are colored by coding system, and the shaded band marks joins between pieces that never shared a link.

In [ ]:
t0 = time.time()
tree_raw = girvan_newman_tree(U_raw)
print(f"Girvan–Newman to the end in {time.time() - t0:.1f}s: {len(tree_raw.levels)} levels")

ERA = {"ICD9CM": df.CATEGORICAL[0], "ICD10CM": df.CATEGORICAL[1]}
fig, ax = plt.subplots(figsize=(15, 5))
df.plot_dendrogram(
    tree_raw,
    {c: ERA.get(vocab(c), df.OTHER) for c in U_raw.nodes},
    labels=None,
    ax=ax,
    orientation="top",
)
ax.legend(
    handles=[
        plt.Rectangle((0, 0), 1, 1, color=ERA["ICD9CM"], label="ICD-9 codes (before Oct 2015)"),
        plt.Rectangle((0, 0), 1, 1, color=ERA["ICD10CM"], label="ICD-10 codes (after)"),
    ],
    loc="upper right",
)
ax.set_title(
    f"Raw codes: the tree's top split is the coding system ({U_raw.number_of_nodes()} diagnoses)"
)
fig.tight_layout()
nf.save(fig, figures_dir / "08_dendrogram_raw_codes.png")
plt.show()

## Part 2 — Harmonized: the tables' standard SNOMED concepts

In OMOP, every diagnosis row also carries its **standard concept**: the billing code already translated to SNOMED. For example, ICD-9 401.9 and ICD-10 I10 are both SNOMED 59621000, *essential hypertension*. The graph is rebuilt with **the same rules** on those concepts. This is also how the project's patient graphs are built (notebooks 07, 10).

In [ ]:
t0 = time.time()
concept_db = OmopDatabase(tables)  # OMOP/<standard concept id>
n_raw = len(patient_code_counts(raw_db, list(raw_db), table="condition"))
n_concepts = len(patient_code_counts(concept_db, list(concept_db), table="condition"))
print(f"{n_raw:,} distinct billing codes -> {n_concepts:,} distinct standard concepts")

G_snomed, _ = build_population_trajectory_graph(concept_db)
names_snomed = concept_names(G_snomed.nodes, sources.concept_table)
U_snomed = symmetrize_max(G_snomed)
pieces_snomed = sorted(nx.connected_components(U_snomed), key=len, reverse=True)
print(
    f"built in {time.time() - t0:.0f}s:",
    U_snomed.number_of_nodes(),
    "concepts,",
    U_snomed.number_of_edges(),
    "links",
)
print("piece sizes:", [len(p) for p in pieces_snomed])

**One main piece now.** The era split is gone. The graph is also smaller than the raw-code one (sizes printed above). Merging the two code systems makes each concept more common, so the same relative risk has to clear a higher bar against the population baseline. The links that remain are the ones that hold across both coding systems.

### Where to cut the tree

Each Girvan–Newman level is a candidate grouping. **Modularity** scores each one: how much denser the links are inside groups than chance would give. The most modular level is the natural place to cut. Louvain, an unrelated algorithm, gives an independent check.

In [ ]:
tree = girvan_newman_tree(U_snomed)
best = tree.levels[tree.best_level]
best_q = tree.modularity[tree.best_level]
print(f"most modular level: {len(best)} communities, modularity {best_q:.3f}")

louvain = community_louvain.best_partition(U_snomed, weight="weight", random_state=0)
louvain_groups = [{n for n in U_snomed if louvain[n] == k} for k in set(louvain.values())]
gn_label = {n: i for i, c in enumerate(best) for n in c}
nodes = list(U_snomed.nodes)
ari = adjusted_rand_score([gn_label[n] for n in nodes], [louvain[n] for n in nodes])
louvain_mod = nx.community.modularity(U_snomed, louvain_groups, weight="weight")
print(f"Louvain: {len(louvain_groups)} communities, modularity {louvain_mod:.3f}")
print(f"agreement with Girvan–Newman (ARI): {ari:.2f}")

fig, ax = plt.subplots(figsize=(9, 3.4))
counts = [len(level) for level in tree.levels]
ax.plot(counts, tree.modularity, color=df.CATEGORICAL[0], lw=2)
ax.plot(
    counts[tree.best_level], tree.modularity[tree.best_level], "o", color=df.CATEGORICAL[0], ms=9
)
ax.annotate(
    f"best: {counts[tree.best_level]} communities, Q = {tree.modularity[tree.best_level]:.3f}",
    (counts[tree.best_level], tree.modularity[tree.best_level]),
    xytext=(25, -120),
    textcoords="offset points",
    color=nf.INK_2,
    arrowprops={"arrowstyle": "-", "color": nf.INK_3, "lw": 0.8},
)
ax.set_xlabel("number of communities (Girvan–Newman removes links left to right)")
ax.set_ylabel("modularity Q")
ax.set_title("Where to cut the tree: modularity at every level")
fig.tight_layout()
nf.save(fig, figures_dir / "08_modularity_by_level.png")
plt.show()

### The dendrogram

Each leaf is one SNOMED concept. Colors mark the seven largest communities at the most modular cut. Smaller groups and singletons are gray, and so are links that join different communities. Read it right to left: diagnoses that stay joined until late splits belong to the same family.

In [ ]:
colors = df.community_colors(tree)
fig, ax = plt.subplots(figsize=(11, 15))
df.plot_dendrogram(tree, colors, labels=names_snomed, ax=ax, orientation="left")
ax.legend(
    handles=df.community_legend(tree, names_snomed, U_snomed),
    title="community size: best-connected member",
    loc="upper left",
    bbox_to_anchor=(0.0, -0.04),
    ncol=2,
    fontsize=8,
    title_fontsize=8,
)
ax.set_title("Disease-trajectory families (SNOMED-harmonized, Girvan–Newman)")
fig.tight_layout()
nf.save(fig, figures_dir / "08_dendrogram_snomed.png", dpi=120)
plt.show()

In [ ]:
# The communities at the best cut, largest first: concept names only.
pd.DataFrame(
    [
        {
            "size": len(c),
            "members (most connected first)": "; ".join(
                names_snomed.get(n, n) for n in sorted(c, key=lambda n: -U_snomed.degree(n))
            ),
        }
        for c in sorted(best, key=len, reverse=True)
        if len(c) > 1
    ]
)

## Save the aggregate results

In [ ]:
rows = []
for condition, U, t in (("raw_codes", U_raw, tree_raw), ("snomed_harmonized", U_snomed, tree)):
    pieces = sorted(nx.connected_components(U), key=len, reverse=True)
    metrics = {
        "n_nodes": U.number_of_nodes(),
        "n_edges": U.number_of_edges(),
        "n_components": len(pieces),
        "largest_cc_size": len(pieces[0]),
        "cross_vocabulary_edges": sum(1 for u, v in U.edges() if vocab(u) != vocab(v)),
        "gn_levels": len(t.levels),
        "gn_forced_joins": t.n_forced_joins,
        "gn_best_n_communities": len(t.levels[t.best_level]),
        "gn_best_modularity": t.modularity[t.best_level],
    }
    for metric, value in metrics.items():
        rows.append(
            {
                "task": "population",
                "condition": condition,
                "graph_variant": GRAPH_VARIANT,
                "metric": metric,
                "value": float(value),
            }
        )
for metric, value in (
    ("louvain_n_communities", len(louvain_groups)),
    ("louvain_modularity", louvain_mod),
    ("ari_gn_best_vs_louvain", ari),
):
    rows.append(
        {
            "task": "population",
            "condition": "snomed_harmonized",
            "graph_variant": GRAPH_VARIANT,
            "metric": metric,
            "value": float(value),
        }
    )
out = write_aggregate(rows, results_dir / "trajectory_dendrogram.csv")
pd.read_csv(out).pivot(index="metric", columns="condition", values="value")

## Findings

Numbers from the Oct 2, 2026 run on the course OMOP tables (all 6,731 patients).

**1. On billing codes, the tree's top split is the 2015 coding change, not biology.** The raw-code graph (134 diagnoses, 584 links) has two big pieces. One is 69 codes, all ICD-10. The other is 47 codes, all ICD-9. **0 of 584 links join them.** The same clinical story appears twice, once per coding system: ICD-9 "unspecified septicemia" and ICD-10 "sepsis, unspecified organism" never meet. A one-year window almost never spans the October 2015 switch.

**2. On the tables' standard SNOMED concepts, the split disappears.** 18,349 distinct billing codes become 8,757 concepts. Rebuilt with identical rules, the graph has **one main piece of 60 concepts** (79 concepts and 142 links in all). It is smaller because merged concepts are more common, so a link must clear a higher baseline. What remains holds across both coding eras.

**3. The harmonized tree finds recognizable clinical families.** The most modular cut has 13 communities (modularity Q = 0.566). The largest groups:
- **pulmonary congestion / abnormal lung findings / respiratory failure** (19)
- **sepsis / septic shock / acute kidney failure** (18)
- **heart failure / chronic pulmonary edema** (14)
- **atrial fibrillation and flutter** (5)
- **pregnancy and birth** (4)
- **diabetes complications** (4)

The smaller pieces are pairs that are effectively one condition coded two ways (vomiting / nausea and vomiting; knee pain / knee osteoarthritis). Girvan–Newman correctly leaves them unjoined.

**4. Louvain broadly agrees.** It finds 13 communities with modularity 0.590, against Girvan–Newman's 0.566. Agreement is ARI 0.65: the same big families with somewhat different borders.

### What this means for the project

- **Coding-system harmonization matters.** Graphs on raw billing codes split by calendar era. The course tables' standard concepts fix that, which is one reason they are the project's primary data source. Notebook 11 measures the same era split in the patient graphs, and shows era explains almost none of the homophily.
- **For the report,** this is a clean, visual example of a data artifact masquerading as structure, caught by classical network analysis.

### Caveats

- Symmetrizing with the stronger direction discards which diagnosis comes first. The tree groups diagnoses that occur near each other, not cause and effect.
- Merge heights are Girvan–Newman step numbers, an ordering rather than a distance.

### Done when
- [x] #14: a real multi-level Girvan–Newman dendrogram (all levels, converted to a scipy linkage matrix), cut at the most modular level, compared with Louvain
- [x] #37: built from the course OMOP tables alone
- [x] Aggregate output: `results/aggregate/trajectory_dendrogram.csv` and figures `08_*.png`